In [1]:
import numpy as np
import json
import matplotlib.pyplot as plt
import os
import pandas as pd
import sys

In [2]:
def masked_xcorr(a, b, va, vb, max_lag, min_overlap=60):
    """Positive lag => a is delayed relative to b (a[i+lag] matches b[i])."""
    na, nb = len(a), len(b)
    out = []
    for lag in range(-max_lag, max_lag + 1):
        # i indexes b; a index is i+lag. Valid i range:
        lo, hi = max(0, -lag), min(nb, na - lag) # getting bounds of b data to be used in correlation
        if hi - lo < min_overlap:   # skip if less than min_overlap points to compare
            continue
        y,  my = b[lo:hi],           vb[lo:hi]
        x,  mx = a[lo+lag:hi+lag],   va[lo+lag:hi+lag]
        m = mx & my                 # mask for valid point in BOTH a segment and b segment
        if m.sum() < min_overlap:   # skip if number of VALID points is less than min_overlap
            continue
        xv, yv = x[m] - x[m].mean(), y[m] - y[m].mean() # extracting normalised valid values from a segment and b segment
        denom = np.sqrt((xv**2).sum() * (yv**2).sum()) # pearson correlation is sum of ab/sqrt(a2*b2)
        if denom > 0:
            out.append((lag, float((xv*yv).sum() / denom), int(m.sum())))
    return out


def make_signal(xy, conf, thresh=0.3):
    valid = conf > thresh                                  # (T, 25)
    step = np.linalg.norm(np.diff(xy, axis=0), axis=2)     # (T-1, 25) getting magnitude of frame-to-frame y move
    ok = valid[1:] & valid[:-1]                            # (T-1, 25) mask for when both previous and next xy values are valid
    w = np.where(ok, conf[1:], 0.0)                        # (T-1, 25) using ok mask to 0 out conf values for any points failing the ok test
    denom = w.sum(axis=1)                                  # (T-1,) sum of valid conf values across each frame
    sig = np.where(denom > 0, (step * w).sum(axis=1) / np.maximum(denom, 1e-9), 0.0) # getting confidence-weighted average joints speed
    v = denom > 0                                          # the validity mask
    sig[v] = (sig[v] - sig[v].mean()) / sig[v].std()       # normalising valid values in sig
    return sig, v

def best_lag(res, min_corr=0.3):
    if not res:
        return None
    lag, corr, n = max(res, key=lambda r: r[1])
    return (lag, corr, n) if corr >= min_corr else None

In [3]:
datapath = '/Volumes/Expansion/MotorDevelopment/Korea/B'
savepath = '/Volumes/Expansion/MotorDevelopment/Korea/B/Aligned'

allJsons = os.listdir(datapath)

allJsons = [f.split('.')[0] for f in allJsons if not f.startswith('.') and f.endswith('.json')  ]
allJsons = ['_'.join(f.split('_')[:-1]) for f in allJsons] # removing camera number

allJsons = list(dict.fromkeys(allJsons))

fileData = pd.DataFrame()

#extracting field info from filename
fileData['FileStub'] = [f for f in allJsons]
fileData['User'] = [f.split('_')[0] for f in allJsons]
fileData['Action'] = [f.split('_')[2] for f in allJsons]
fileData['Rep'] = [f.split('_')[3] for f in allJsons]

fileData['Cam1Filepath'] = fileData['FileStub'].apply(lambda x: x + '_1.json')
fileData['Cam2Filepath'] = fileData['FileStub'].apply(lambda x: x + '_2.json')
fileData['Cam3Filepath'] = fileData['FileStub'].apply(lambda x: x + '_3.json')
fileData['allCamsPresent'] = True

fileData['Lag12'] = np.nan
fileData['Lag13'] = np.nan
fileData['Lag23'] = np.nan
fileData['Lag23_calc'] = np.nan
fileData['LagCheck'] = False
fileData['Status'] = 'not_processed'
fileData['Error'] = ''

In [4]:
numRows = len(allJsons)

for idx, fieldVals in fileData.iterrows():
    
   
    sys.stdout.write(f'\r{idx}/{numRows}')
    

    filepath_cam1 = os.path.join(datapath,fieldVals['Cam1Filepath'])
    if not os.path.exists(filepath_cam1):
        fileData.loc[idx, 'allCamsPresent'] = False
        fileData.loc[idx, 'Status'] = 'Missing Cam 1'
        continue
    with open(filepath_cam1) as f:
        d1 = json.load(f)

    filepath_cam2 = os.path.join(datapath,fieldVals['Cam2Filepath'])
    if not os.path.exists(filepath_cam2):
        fileData.loc[idx, 'allCamsPresent'] = False
        fileData.loc[idx, 'Status'] = 'Missing Cam 2'
        continue
    with open(filepath_cam2) as f:
        d2 = json.load(f)

    filepath_cam3 = os.path.join(datapath,fieldVals['Cam3Filepath'])
    if  not os.path.exists(filepath_cam3):
        fileData.loc[idx, 'allCamsPresent'] = False
        fileData.loc[idx, 'Status'] = 'Missing Cam 3'
        continue
    with open(filepath_cam3) as f:
        d3 = json.load(f)

    try:
        xy1 = np.array([fr['skeleton'][0]['pose'] for fr in d1['data']]).reshape(-1, 25, 2)
        score1 = np.array([fr['skeleton'][0]['score'] for fr in d1['data']])

        xy2 = np.array([fr['skeleton'][0]['pose'] for fr in d2['data']]).reshape(-1, 25, 2)
        score2 = np.array([fr['skeleton'][0]['score'] for fr in d2['data']])

        xy3 = np.array([fr['skeleton'][0]['pose'] for fr in d3['data']]).reshape(-1, 25, 2)
        score3 = np.array([fr['skeleton'][0]['score'] for fr in d3['data']])
    except Exception as e:
        fileData.loc[idx, 'Status'] = 'Empty skeleton list'
        fileData.loc[idx, 'Error'] = repr(e)
        continue

    try:
        sig1, v1 = make_signal(xy1, score1)
        sig2, v2 = make_signal(xy2, score2)
        sig3, v3 = make_signal(xy3, score3)

        b12 = best_lag(masked_xcorr(sig1, sig2, v1, v2, 30))
        b13 = best_lag(masked_xcorr(sig1, sig3, v1, v3, 30))
        b23 = best_lag(masked_xcorr(sig2, sig3, v2, v3, 30))

        if any(b is None for b in (b12, b13, b23)):
            fileData.loc[idx, 'LagCheck'] = False
            fileData.loc[idx, 'Status'] = 'Failed lag check'
            continue
        
        fileData.loc[idx, ['Lag12','Lag13','Lag23']] = b12[0], b13[0], b23[0]
        fileData.loc[idx, 'Lag23_calc'] = b13[0] - b12[0]
        if abs(b23[0] - (b13[0] - b12[0])) <= 1:
            fileData.loc[idx, 'LagCheck'] = True
            
        else:
            fileData.loc[idx, 'LagCheck'] = False
            fileData.loc[idx, 'Status'] = 'LagCheck False'
            continue

    except Exception as e:
        fileData.loc[idx, 'Status'] = 'Lag stage error'
        fileData.loc[idx, 'Error'] = repr(e)
        continue

    
    fileData.loc[idx, 'Status'] = 'OK'


    

771/1555

/var/folders/w5/38gvs5991kl2xwv6ry5ys4mw0000gn/T/ipykernel_1306/1662832057.py:30: RuntimeWarning: Mean of empty slice.
  sig[v] = (sig[v] - sig[v].mean()) / sig[v].std()       # normalising valid values in sig
/Users/peterkearney/anaconda3/envs/motEnv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/peterkearney/anaconda3/envs/motEnv/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/peterkearney/anaconda3/envs/motEnv/lib/python3.11/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/peterkearney/anaconda3/envs/motEnv/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(

1554/1555

In [5]:
for idx, fieldVals in fileData.iterrows():
    
    sys.stdout.write(f'\r{idx}/{numRows}')

    # need to skip bad reps

    if fieldVals['Status'] != 'OK':
        continue

    filepath_cam1 = os.path.join(datapath,fieldVals['Cam1Filepath'])
    with open(filepath_cam1) as f:
        d1 = json.load(f)

    filepath_cam2 = os.path.join(datapath,fieldVals['Cam2Filepath'])
    with open(filepath_cam2) as f:
        d2 = json.load(f)

    filepath_cam3 = os.path.join(datapath,fieldVals['Cam3Filepath'])
    with open(filepath_cam3) as f:
        d3 = json.load(f)

    
    xy1 = np.array([fr['skeleton'][0]['pose'] for fr in d1['data']]).reshape(-1, 25, 2)
    score1 = np.array([fr['skeleton'][0]['score'] for fr in d1['data']])

    xy2 = np.array([fr['skeleton'][0]['pose'] for fr in d2['data']]).reshape(-1, 25, 2)
    score2 = np.array([fr['skeleton'][0]['score'] for fr in d2['data']])

    xy3 = np.array([fr['skeleton'][0]['pose'] for fr in d3['data']]).reshape(-1, 25, 2)
    score3 = np.array([fr['skeleton'][0]['score'] for fr in d3['data']])

    # syncing cameras 1 and 2 first
    lag12 = int(fieldVals['Lag12'])
    lag13 = int(fieldVals['Lag13'])

    if lag12 >0:
        xy1 = xy1[lag12:]
        score1 = score1[lag12:]
        lag13 = lag13-lag12
    else:
        xy2 = xy2[-lag12:]
        score2 = score2[-lag12:]


    if lag13 > 0:
        xy1 = xy1[lag13:]
        score1 = score1[lag13:]
        xy2 = xy2[lag13:]
        score2 = score2[lag13:]
    else:
        xy3 = xy3[-lag13:]
        score3 = score3[-lag13:]

    min_length = min(xy1.shape[0],xy2.shape[0],xy3.shape[0])

    xy1 = xy1[:min_length]
    score1 = score1[:min_length]
    xy2 = xy2[:min_length]
    score2 = score2[:min_length]
    xy3 = xy3[:min_length]
    score3 = score3[:min_length]


    filename = fieldVals['FileStub'] + '.npz'
    filepath = os.path.join(savepath,filename)
    np.savez(filepath,xy1 = xy1, xy2 = xy2, xy3 = xy3, score1 = score1, score2 = score2, score3 = score3)


    



    


1554/1555